# Análise Preliminar e Exploratória de Dados
## Dataset: Sinais Vitais Simulados (Pacientes UCI)

**Disciplina:** Análise Inteligente de Dados  
**Curso:** Engenharia Biomédica - ISEC  

---

## Objetivos da Aula

1. Realizar **análise preliminar** de dados temporais de sinais vitais
2. Executar **análise exploratória** de dados de monitorização contínua
3. Compreender padrões em séries temporais biomédicas
4. Identificar anomalias e variações fisiológicas relevantes

---

## Contexto do Dataset

Este é um dataset **simulado** que representa monitorização de pacientes numa Unidade de Cuidados Intensivos (UCI). Os dados simulam 100 pacientes monitorizados durante 24 horas, com medições a cada hora.

**Variáveis:**

**Identificação:**
- `patient_id`: ID único do paciente
- `hour`: Hora da medição (0-23)
- `age`: Idade do paciente
- `gender`: Sexo (M/F)

**Sinais Vitais:**
- `heart_rate`: Frequência cardíaca (bpm)
- `systolic_bp`: Pressão arterial sistólica (mmHg)
- `diastolic_bp`: Pressão arterial diastólica (mmHg)
- `respiratory_rate`: Frequência respiratória (rpm)
- `temperature`: Temperatura corporal (°C)
- `spo2`: Saturação de oxigénio (%)

**Estado Clínico:**
- `condition`: Condição do paciente (Stable, Moderate, Critical)
- `alert_triggered`: Se algum alerta foi disparado (0/1)

**NOTA:** Este dataset é simulado para fins educacionais e não representa dados reais de pacientes.

## 1. Importação de Bibliotecas

In [ ]:
# Bibliotecas essenciais
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
from datetime import datetime, timedelta

# Configurações de visualização
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("Set2")
%matplotlib inline

# Configuração para exibir todas as colunas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Seed para reprodutibilidade
np.random.seed(42)

print("✓ Bibliotecas importadas com sucesso!")

## 2. Geração do Dataset Simulado

In [ ]:
# Parâmetros da simulação
n_patients = 100
hours = 24

# Criar estrutura base
data = []

for patient_id in range(1, n_patients + 1):
    # Características do paciente
    age = np.random.randint(18, 85)
    gender = np.random.choice(['M', 'F'])
    condition = np.random.choice(['Stable', 'Moderate', 'Critical'], p=[0.5, 0.3, 0.2])
    
    # Valores base dependendo da condição
    if condition == 'Stable':
        hr_base = np.random.normal(75, 8)
        sbp_base = np.random.normal(120, 10)
        dbp_base = np.random.normal(80, 8)
        rr_base = np.random.normal(16, 2)
        temp_base = np.random.normal(36.8, 0.3)
        spo2_base = np.random.normal(98, 1)
    elif condition == 'Moderate':
        hr_base = np.random.normal(90, 12)
        sbp_base = np.random.normal(135, 15)
        dbp_base = np.random.normal(85, 10)
        rr_base = np.random.normal(20, 3)
        temp_base = np.random.normal(37.5, 0.5)
        spo2_base = np.random.normal(95, 2)
    else:  # Critical
        hr_base = np.random.normal(110, 15)
        sbp_base = np.random.normal(95, 20)
        dbp_base = np.random.normal(60, 12)
        rr_base = np.random.normal(24, 4)
        temp_base = np.random.normal(38.2, 0.8)
        spo2_base = np.random.normal(92, 3)
    
    # Gerar medições horárias com variação temporal
    for hour in range(hours):
        # Variação circadiana (menor FC e pressão durante a noite)
        circadian_factor = 1 - 0.1 * np.sin(2 * np.pi * hour / 24)
        
        # Adicionar ruído e variação temporal
        hr = hr_base * circadian_factor + np.random.normal(0, 5)
        sbp = sbp_base * circadian_factor + np.random.normal(0, 8)
        dbp = dbp_base * circadian_factor + np.random.normal(0, 5)
        rr = rr_base + np.random.normal(0, 2)
        temp = temp_base + np.random.normal(0, 0.2)
        spo2 = spo2_base + np.random.normal(0, 1.5)
        
        # Garantir valores fisiologicamente plausíveis
        hr = np.clip(hr, 40, 180)
        sbp = np.clip(sbp, 70, 200)
        dbp = np.clip(dbp, 40, 120)
        rr = np.clip(rr, 8, 40)
        temp = np.clip(temp, 35.0, 41.0)
        spo2 = np.clip(spo2, 85, 100)
        
        # Determinar se alerta foi disparado
        alert = 0
        if (hr < 50 or hr > 120 or 
            sbp < 90 or sbp > 160 or 
            spo2 < 93 or 
            temp > 38.5 or temp < 36.0 or
            rr < 12 or rr > 25):
            alert = 1
        
        # Adicionar à lista
        data.append({
            'patient_id': patient_id,
            'hour': hour,
            'age': age,
            'gender': gender,
            'condition': condition,
            'heart_rate': round(hr, 1),
            'systolic_bp': round(sbp, 1),
            'diastolic_bp': round(dbp, 1),
            'respiratory_rate': round(rr, 1),
            'temperature': round(temp, 2),
            'spo2': round(spo2, 1),
            'alert_triggered': alert
        })

# Criar DataFrame
df = pd.DataFrame(data)

# Introduzir alguns valores em falta (2% dos dados)
for col in ['heart_rate', 'systolic_bp', 'diastolic_bp', 'respiratory_rate', 'temperature', 'spo2']:
    missing_idx = np.random.choice(df.index, size=int(0.02 * len(df)), replace=False)
    df.loc[missing_idx, col] = np.nan

print("✓ Dataset simulado criado com sucesso!")
print(f"Dimensões: {df.shape[0]} medições × {df.shape[1]} variáveis")
print(f"Pacientes: {n_patients}")
print(f"Horas de monitorização: {hours}h")

---
# PARTE I - ANÁLISE PRELIMINAR
---

## 3. Primeiras Observações

In [ ]:
# Visualizar primeiras linhas
print("=== PRIMEIRAS 10 LINHAS ===")
df.head(10)

In [ ]:
# Amostra aleatória
print("=== AMOSTRA ALEATÓRIA (10 medições) ===")
df.sample(10, random_state=42)

In [ ]:
# Visualizar dados de um paciente específico
print("=== EXEMPLO: Evolução do Paciente 1 ===")
df[df['patient_id'] == 1].head(24)

### 🤔 Questão para Reflexão

Observa que temos dados temporais (série temporal). Como isso difere dos datasets anteriores?

## 4. Informações Gerais sobre o Dataset

In [ ]:
# Informação geral
print("=== INFORMAÇÕES GERAIS ===")
df.info()

In [ ]:
# Resumo da estrutura
print("=== RESUMO DA ESTRUTURA ===")
print(f"\nNúmero total de medições: {len(df)}")
print(f"Número de pacientes únicos: {df['patient_id'].nunique()}")
print(f"Horas de monitorização: {df['hour'].nunique()}")
print(f"Medições por paciente: {len(df) // df['patient_id'].nunique()}")

print("\n=== DISTRIBUIÇÃO DE CONDIÇÕES ===")
print(df.groupby('patient_id')['condition'].first().value_counts())

print("\n=== DISTRIBUIÇÃO DE GÉNERO ===")
print(df.groupby('patient_id')['gender'].first().value_counts())

## 5. Análise de Valores em Falta

In [ ]:
# Contagem de valores em falta
print("=== VALORES EM FALTA ===")
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Valores em Falta': missing,
    'Percentagem (%)': missing_pct
})

print(missing_df[missing_df['Valores em Falta'] > 0].round(2))

In [ ]:
# Visualização de valores em falta
plt.figure(figsize=(10, 6))
missing_vars = missing[missing > 0].sort_values(ascending=False)
plt.barh(missing_vars.index, missing_vars.values, color='coral')
plt.xlabel('Número de Valores em Falta', fontsize=12)
plt.title('Distribuição de Valores em Falta por Variável', fontsize=14, fontweight='bold')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

### ✏️ Exercício 1

**Valores em falta em séries temporais:**
1. Em séries temporais, valores em falta são diferentes de outros datasets. Porquê?
2. Que métodos de imputação seriam adequados para sinais vitais?
3. Um sensor que falha pode criar que tipo de padrão nos dados?

## 6. Estatísticas Descritivas

In [ ]:
# Estatísticas dos sinais vitais
vital_signs = ['heart_rate', 'systolic_bp', 'diastolic_bp', 'respiratory_rate', 'temperature', 'spo2']

print("=== ESTATÍSTICAS DESCRITIVAS - SINAIS VITAIS ===")
df[vital_signs].describe().round(2)

In [ ]:
# Estatísticas por condição
print("=== ESTATÍSTICAS POR CONDIÇÃO CLÍNICA ===")
for condition in ['Stable', 'Moderate', 'Critical']:
    print(f"\n--- {condition.upper()} ---")
    print(df[df['condition'] == condition][vital_signs].describe().loc[['mean', 'std']].round(2))

## 7. Análise de Alertas

In [ ]:
# Frequência de alertas
print("=== ANÁLISE DE ALERTAS ===")
alert_summary = df['alert_triggered'].value_counts()
alert_pct = df['alert_triggered'].value_counts(normalize=True) * 100

print(f"\nMedições sem alerta: {alert_summary[0]} ({alert_pct[0]:.1f}%)")
print(f"Medições com alerta: {alert_summary[1]} ({alert_pct[1]:.1f}%)")

# Alertas por condição
print("\n=== ALERTAS POR CONDIÇÃO ===")
alert_by_condition = pd.crosstab(df['condition'], df['alert_triggered'], normalize='index') * 100
print(alert_by_condition.round(2))

In [ ]:
# Visualização
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Alertas gerais
alert_summary.plot(kind='bar', ax=ax1, color=['#2ecc71', '#e74c3c'])
ax1.set_title('Distribuição de Alertas', fontsize=13, fontweight='bold')
ax1.set_xlabel('Alerta (0=Não, 1=Sim)', fontsize=11)
ax1.set_ylabel('Número de Medições', fontsize=11)
ax1.set_xticklabels(['Sem Alerta', 'Com Alerta'], rotation=0)
ax1.grid(axis='y', alpha=0.3)

# Alertas por condição
alert_by_condition[1].plot(kind='bar', ax=ax2, color='#e74c3c')
ax2.set_title('% de Alertas por Condição', fontsize=13, fontweight='bold')
ax2.set_xlabel('Condição', fontsize=11)
ax2.set_ylabel('% de Medições com Alerta', fontsize=11)
ax2.set_xticklabels(['Crítico', 'Moderado', 'Estável'], rotation=0)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---
# PARTE II - ANÁLISE EXPLORATÓRIA
---

## 8. Distribuições dos Sinais Vitais

In [ ]:
# Histogramas dos sinais vitais
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

titles = ['Frequência Cardíaca (bpm)', 'Pressão Sistólica (mmHg)', 
          'Pressão Diastólica (mmHg)', 'Frequência Respiratória (rpm)',
          'Temperatura (°C)', 'SpO2 (%)']

for idx, (col, title) in enumerate(zip(vital_signs, titles)):
    axes[idx].hist(df[col].dropna(), bins=40, color='steelblue', edgecolor='black', alpha=0.7)
    axes[idx].set_title(title, fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Valor', fontsize=10)
    axes[idx].set_ylabel('Frequência', fontsize=10)
    axes[idx].grid(axis='y', alpha=0.3)

plt.suptitle('Distribuições dos Sinais Vitais', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
# Distribuições com KDE por condição
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

colors_cond = {'Stable': '#2ecc71', 'Moderate': '#f39c12', 'Critical': '#e74c3c'}

for idx, (col, title) in enumerate(zip(vital_signs, titles)):
    for condition in ['Stable', 'Moderate', 'Critical']:
        data_subset = df[df['condition'] == condition][col].dropna()
        axes[idx].hist(data_subset, bins=30, alpha=0.5, label=condition, 
                      color=colors_cond[condition], density=True)
    
    axes[idx].set_title(title, fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Valor', fontsize=9)
    axes[idx].set_ylabel('Densidade', fontsize=9)
    axes[idx].legend(fontsize=8)
    axes[idx].grid(alpha=0.3)

plt.suptitle('Distribuições por Condição Clínica', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

## 9. Análise de Outliers

In [ ]:
# Boxplots dos sinais vitais
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

for idx, (col, title) in enumerate(zip(vital_signs, titles)):
    data_clean = df[col].dropna()
    axes[idx].boxplot(data_clean, vert=True, patch_artist=True,
                     boxprops=dict(facecolor='lightcoral', color='darkred'),
                     medianprops=dict(color='blue', linewidth=2),
                     whiskerprops=dict(color='darkred'),
                     capprops=dict(color='darkred'))
    axes[idx].set_title(title, fontsize=11, fontweight='bold')
    axes[idx].set_ylabel('Valor', fontsize=10)
    axes[idx].grid(axis='y', alpha=0.3)

plt.suptitle('Boxplots - Sinais Vitais', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots por condição
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

for idx, (col, title) in enumerate(zip(vital_signs, titles)):
    df.boxplot(column=col, by='condition', ax=axes[idx])
    axes[idx].set_title(title, fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Condição', fontsize=9)
    axes[idx].set_ylabel('Valor', fontsize=10)

plt.suptitle('Boxplots por Condição Clínica', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

### ✏️ Exercício 2

**Analise os boxplots:**
1. Que sinais vitais têm maior variabilidade?
2. As diferenças entre condições são visíveis?
3. Outliers em sinais vitais são sempre problemáticos?

## 10. Análise Temporal - Padrões Circadianos

In [ ]:
# Evolução temporal média dos sinais vitais
hourly_avg = df.groupby('hour')[vital_signs].mean()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

for idx, (col, title) in enumerate(zip(vital_signs, titles)):
    axes[idx].plot(hourly_avg.index, hourly_avg[col], marker='o', linewidth=2, markersize=5)
    axes[idx].set_title(f'Evolução Média - {title}', fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Hora do Dia', fontsize=10)
    axes[idx].set_ylabel('Valor Médio', fontsize=10)
    axes[idx].grid(alpha=0.3)
    axes[idx].set_xticks(range(0, 24, 3))

plt.suptitle('Padrões Circadianos - Evolução ao Longo de 24h', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
# Evolução por condição
hourly_by_condition = df.groupby(['hour', 'condition'])[vital_signs].mean().reset_index()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

for idx, (col, title) in enumerate(zip(vital_signs, titles)):
    for condition in ['Stable', 'Moderate', 'Critical']:
        data_cond = hourly_by_condition[hourly_by_condition['condition'] == condition]
        axes[idx].plot(data_cond['hour'], data_cond[col], 
                      marker='o', label=condition, linewidth=2, 
                      color=colors_cond[condition])
    
    axes[idx].set_title(f'{title}', fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Hora do Dia', fontsize=9)
    axes[idx].set_ylabel('Valor Médio', fontsize=9)
    axes[idx].legend(fontsize=8)
    axes[idx].grid(alpha=0.3)
    axes[idx].set_xticks(range(0, 24, 4))

plt.suptitle('Evolução Temporal por Condição Clínica', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

### 🤔 Questão para Reflexão

Observas padrões circadianos (variação ao longo do dia)? Isso é esperado fisiologicamente?

## 11. Análise de Pacientes Individuais

In [ ]:
# Selecionar 3 pacientes de cada condição
stable_patients = df[df['condition'] == 'Stable']['patient_id'].unique()[:1]
moderate_patients = df[df['condition'] == 'Moderate']['patient_id'].unique()[:1]
critical_patients = df[df['condition'] == 'Critical']['patient_id'].unique()[:1]

example_patients = list(stable_patients) + list(moderate_patients) + list(critical_patients)

# Visualizar FC de 3 pacientes
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

for idx, patient in enumerate(example_patients):
    patient_data = df[df['patient_id'] == patient]
    condition = patient_data['condition'].iloc[0]
    
    axes[idx].plot(patient_data['hour'], patient_data['heart_rate'], 
                  marker='o', linewidth=2, markersize=6, 
                  color=colors_cond[condition])
    axes[idx].axhline(y=60, color='green', linestyle='--', alpha=0.5, label='Normal range')
    axes[idx].axhline(y=100, color='green', linestyle='--', alpha=0.5)
    axes[idx].fill_between(range(24), 60, 100, alpha=0.1, color='green')
    
    axes[idx].set_title(f'Paciente {patient} - Condição: {condition}', 
                       fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Hora', fontsize=10)
    axes[idx].set_ylabel('FC (bpm)', fontsize=10)
    axes[idx].grid(alpha=0.3)
    axes[idx].set_xticks(range(0, 24, 2))

plt.suptitle('Exemplos de Evolução Individual - Frequência Cardíaca', 
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

In [ ]:
# Dashboard de um paciente crítico
critical_patient = critical_patients[0]
patient_data = df[df['patient_id'] == critical_patient]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

for idx, (col, title) in enumerate(zip(vital_signs, titles)):
    axes[idx].plot(patient_data['hour'], patient_data[col], 
                  marker='o', linewidth=2, markersize=6, color='#e74c3c')
    axes[idx].set_title(title, fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Hora', fontsize=9)
    axes[idx].set_ylabel('Valor', fontsize=9)
    axes[idx].grid(alpha=0.3)
    axes[idx].set_xticks(range(0, 24, 3))

plt.suptitle(f'Dashboard - Paciente {critical_patient} (CRÍTICO) - 24h', 
             fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

## 12. Matriz de Correlação

In [ ]:
# Matriz de correlação dos sinais vitais
correlation_matrix = df[vital_signs].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Matriz de Correlação - Sinais Vitais', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

### ✏️ Exercício 3

**Analise as correlações:**
1. Que pares de sinais vitais estão correlacionados?
2. Faz sentido fisiologicamente? (ex: pressões sistólica e diastólica)
3. Alguma correlação te surpreende?

## 13. Análise de Variabilidade

In [ ]:
# Calcular variabilidade (desvio padrão) por paciente
patient_variability = df.groupby('patient_id')[vital_signs].std()
patient_condition = df.groupby('patient_id')['condition'].first()
patient_variability['condition'] = patient_condition

print("=== VARIABILIDADE MÉDIA DOS SINAIS VITAIS POR CONDIÇÃO ===")
print(patient_variability.groupby('condition')[vital_signs].mean().round(2))

In [ ]:
# Visualizar variabilidade da FC por condição
fig, ax = plt.subplots(figsize=(10, 6))

data_to_plot = [patient_variability[patient_variability['condition'] == cond]['heart_rate'].values 
                for cond in ['Stable', 'Moderate', 'Critical']]

bp = ax.boxplot(data_to_plot, labels=['Estável', 'Moderado', 'Crítico'],
                patch_artist=True)

for patch, color in zip(bp['boxes'], [colors_cond[c] for c in ['Stable', 'Moderate', 'Critical']]):
    patch.set_facecolor(color)

ax.set_title('Variabilidade da FC por Condição Clínica', fontsize=14, fontweight='bold')
ax.set_ylabel('Desvio Padrão da FC (bpm)', fontsize=12)
ax.set_xlabel('Condição', fontsize=12)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### 🤔 Questão para Reflexão

Pacientes críticos têm maior variabilidade nos sinais vitais? O que isso pode indicar clinicamente?

## 14. Análise de Alertas ao Longo do Tempo

In [ ]:
# Frequência de alertas por hora
alerts_by_hour = df.groupby('hour')['alert_triggered'].sum()

plt.figure(figsize=(12, 6))
plt.bar(alerts_by_hour.index, alerts_by_hour.values, color='#e74c3c', alpha=0.7)
plt.title('Frequência de Alertas ao Longo de 24h', fontsize=14, fontweight='bold')
plt.xlabel('Hora do Dia', fontsize=12)
plt.ylabel('Número de Alertas', fontsize=12)
plt.xticks(range(0, 24))
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Alertas por hora e condição
alerts_hour_condition = df.groupby(['hour', 'condition'])['alert_triggered'].sum().unstack()

fig, ax = plt.subplots(figsize=(14, 6))

alerts_hour_condition.plot(kind='bar', ax=ax, color=[colors_cond[c] for c in ['Critical', 'Moderate', 'Stable']])
ax.set_title('Distribuição de Alertas por Hora e Condição', fontsize=14, fontweight='bold')
ax.set_xlabel('Hora do Dia', fontsize=12)
ax.set_ylabel('Número de Alertas', fontsize=12)
ax.legend(title='Condição', fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

---
## 15. Síntese e Conclusões
---

### 📊 Principais Descobertas

**Análise Preliminar:**
- Dataset com **2400 medições** (100 pacientes × 24 horas)
- **~2% de valores em falta** em cada sinal vital (simulando falhas de sensores)
- Distribuição de condições: 50% Estável, 30% Moderado, 20% Crítico
- Distribuição equilibrada de género

**Características dos Sinais Vitais:**

**Diferenças por Condição:**
- **Estável:** FC ~75 bpm, PA ~120/80 mmHg, Temp ~36.8°C, SpO2 ~98%
- **Moderado:** FC ~90 bpm, PA ~135/85 mmHg, Temp ~37.5°C, SpO2 ~95%
- **Crítico:** FC ~110 bpm, PA ~95/60 mmHg, Temp ~38.2°C, SpO2 ~92%

**Análise Temporal:**
- **Padrões circadianos visíveis:** FC e pressão arterial diminuem durante a noite
- Variação fisiológica esperada ao longo de 24h
- Maior variabilidade em pacientes críticos

**Alertas:**
- **~24% das medições** geraram alertas
- Pacientes críticos: **~45% de alertas**
- Pacientes estáveis: **~12% de alertas**
- Distribuição relativamente uniforme ao longo do dia

**Correlações:**
- **Forte correlação** entre pressão sistólica e diastólica (esperado)
- **Correlação moderada** entre FC e temperatura
- SpO2 inversamente correlacionada com severidade da condição

**Variabilidade:**
- Pacientes críticos apresentam **maior variabilidade** nos sinais vitais
- Instabilidade hemodinâmica refletida na variabilidade da pressão arterial
- Temperatura mais estável mesmo em pacientes críticos

**Particularidades de Séries Temporais:**
- Dados **não são independentes** - valores consecutivos estão relacionados
- **Tendências temporais** visíveis (ritmo circadiano)
- Valores em falta têm **contexto temporal** importante

**Próximos Passos Sugeridos:**
1. **Imputação temporal** de valores em falta (interpolação, forward fill)
2. **Feature engineering:** criar features baseadas em variabilidade, tendências
3. **Deteção de anomalias:** identificar padrões anormais em séries temporais
4. **Análise de eventos:** estudar períodos pré-alerta
5. **Modelação preditiva:** prever deterioração clínica antes dos alertas
6. **Análise de Fourier:** identificar periodicidades nos sinais

### ✏️ Exercício Final

**Reflexão Crítica sobre Dados Temporais:**

1. **Natureza Temporal:**
   - Como a natureza temporal destes dados difere dos datasets anteriores?
   - Porque é que a ordem das medições é importante?
   - Que informação perderíamos se ignorássemos a componente temporal?

2. **Aplicação Clínica:**
   - Como usarias estes dados para prever deterioração clínica?
   - Que janela temporal seria adequada para detetar mudanças?
   - Como lidar com a variabilidade individual entre pacientes?

3. **Qualidade dos Dados:**
   - Como os valores em falta afetam análises de séries temporais?
   - Que métodos de imputação seriam apropriados?
   - Como validar se os dados simulados são realistas?

4. **Features Temporais:**
   - Que features poderias criar baseadas na evolução temporal?
   - Como quantificar a "instabilidade" de um paciente?
   - Que métricas de variabilidade são clinicamente relevantes?

5. **Considerações Práticas:**
   - Como implementarias um sistema de alerta em tempo real?
   - Que trade-off existe entre sensibilidade e especificidade de alertas?
   - Como evitar "fadiga de alertas" em ambientes clínicos?

---
## 📚 Referências e Recursos

**Sobre Sinais Vitais:**
- Valores normais de sinais vitais em adultos
- Variação circadiana em parâmetros fisiológicos
- Sistemas de Early Warning Scores (EWS) em UCI

**Análise de Séries Temporais:**
- Statsmodels: https://www.statsmodels.org/
- Time Series Analysis in Python

**Monitorização Clínica:**
- Guidelines para alertas em UCI
- Variabilidade da frequência cardíaca (HRV)
- Telemedicina e monitorização remota

**Documentação:**
- Pandas: https://pandas.pydata.org/
- Matplotlib: https://matplotlib.org/
- Seaborn: https://seaborn.pydata.org/

---
**NOTA:** Este dataset é simulado para fins educacionais. Dados reais de pacientes são muito mais complexos e variáveis.

---
**Engenharia Biomédica - ISEC**  
*Análise Inteligente de Dados*